# Apache Spark Data Lake Preprocessing - Sales Dataset

**Environment:** Google Colab + PySpark  |  **Dataset:** `sales.csv`

This notebook builds a simple Bronze / Silver data lake:
- **Bronze** - the raw `sales.csv`, kept exactly as provided
- **Silver** - the basic-preprocessed data written by Spark (Parquet + a single CSV for submission)

Run it with **Runtime -> Restart and run all**. When the upload prompt appears, choose `sales.csv`.

## 1. Data Lake structure

In [1]:
import os


if not os.path.exists("sales.csv"):
    from google.colab import files
    uploaded = files.upload()
    name = list(uploaded.keys())[0]
    if name != "sales.csv":          # Colab renames re-uploads, e.g. "sales (1).csv"
        os.rename(name, "sales.csv")

print("Found:", os.path.exists("sales.csv"))

Saving sales.csv to sales.csv
Found: True


In [2]:
import shutil

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

# Copy the original file into Bronze WITHOUT modifying it
shutil.copy("sales.csv", "data_lake/bronze/sales.csv")

def show_tree(root_dir="data_lake"):
    for root, dirs, files_ in os.walk(root_dir):
        level = root.replace(root_dir, "").count(os.sep)
        indent = "    " * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in sorted(files_):
            if not f.startswith("."):
                print(f"{indent}    {f}")

show_tree()

data_lake/
    bronze/
        sales.csv
    silver/
        preprocessed_sales/


## 2. Load and inspect with Spark

In [3]:
!pip install -q pyspark

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = SparkSession.builder.appName("SparkSalesLakeWorkflow").getOrCreate()
print("Spark version:", spark.version)

Spark version: 4.0.4


In [5]:
bronze_path = "data_lake/bronze/sales.csv"

df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv(bronze_path))

bronze_rows = df.count()
print("Total records in Bronze (before preprocessing):", bronze_rows)
df.printSchema()

Total records in Bronze (before preprocessing): 1000
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [6]:
df.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

### 2.1 Duplicates and missing values

In [7]:
duplicate_count = bronze_rows - df.dropDuplicates().count()
print("Exact duplicate rows:", duplicate_count)

df.groupBy(df.columns).count().filter(F.col("count") > 1).show(truncate=False)

Exact duplicate rows: 10
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100692  |C0141      |Varun Agarwal   |Electric Kettle  |Home Appliances|4       |920       |0               |Net Banking     |Delhi     |Delhi         |2026-08-18|Shipped     |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|100297  |C0149      |Manish Pillai   |Dry Fruits Pack 

In [8]:
# Null count per column
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



### 2.2 Text-format inconsistencies

In [9]:
text_cols = ["customer_id", "customer_name", "product", "category",
             "payment_method", "city", "state", "order_status"]

for c in text_cols:
    # values with leading/trailing spaces
    padded = df.filter(F.col(c) != F.trim(F.col(c))).count()
    # values that differ only by case/spacing (e.g. 'beauty' vs 'Beauty')
    variants = (df.filter(F.col(c).isNotNull())
                  .groupBy(F.lower(F.trim(F.col(c))).alias("key"))
                  .agg(F.countDistinct(c).alias("spellings"))
                  .filter(F.col("spellings") > 1)
                  .count())
    print(f"{c:15s} padded values: {padded:3d} | values with >1 spelling: {variants}")

customer_id     padded values:   0 | values with >1 spelling: 0
customer_name   padded values:   3 | values with >1 spelling: 3
product         padded values:   0 | values with >1 spelling: 0
category        padded values:   0 | values with >1 spelling: 4
payment_method  padded values:   0 | values with >1 spelling: 0
city            padded values:   3 | values with >1 spelling: 5
state           padded values:   0 | values with >1 spelling: 0
order_status    padded values:   0 | values with >1 spelling: 0


In [10]:
# Examples of the inconsistent values
for c in ["customer_name", "category", "city"]:
    print(f"--- {c}")
    (df.filter(F.col(c).isNotNull())
       .groupBy(F.lower(F.trim(F.col(c))).alias("key"))
       .agg(F.collect_set(c).alias("spellings"))
       .filter(F.size("spellings") > 1)
       .show(10, truncate=False))

--- customer_name
+---------------+-----------------------------------+
|key            |spellings                          |
+---------------+-----------------------------------+
|akash jain     |[Akash Jain , Akash Jain]          |
|meera joshi    |[Meera Joshi, Meera Joshi ]        |
|tanvi chaudhary|[Tanvi Chaudhary, Tanvi Chaudhary ]|
+---------------+-----------------------------------+

--- category
+-----------+--------------------------+
|key        |spellings                 |
+-----------+--------------------------+
|beauty     |[beauty, Beauty]          |
|electronics|[ELECTRONICS, Electronics]|
|sports     |[sports, Sports, SPORTS]  |
|clothing   |[CLOTHING, Clothing]      |
+-----------+--------------------------+

--- city
+----------+-----------------------------------+
|key       |spellings                          |
+----------+-----------------------------------+
|delhi     |[DELHI, delhi, Delhi]              |
|hyderabad |[HYDERABAD, Hyderabad,  Hyderabad ]|
|mumbai

### 2.3 Invalid numerical values

In [11]:
for c in ["quantity", "unit_price", "discount_percent"]:
    print(c)
    df.select(F.min(c).alias("min"), F.max(c).alias("max"),
              F.round(F.avg(c), 2).alias("avg")).show()

quantity
+---+---+----+
|min|max| avg|
+---+---+----+
| -2| 10|5.36|
+---+---+----+

unit_price
+----+-----+------+
| min|  max|   avg|
+----+-----+------+
|-500|81010|5702.4|
+----+-----+------+

discount_percent
+---+---+-----+
|min|max|  avg|
+---+---+-----+
| -5|120|19.95|
+---+---+-----+



In [12]:
invalid_rules = {
    "quantity":         F.col("quantity") <= 0,
    "unit_price":       F.col("unit_price") <= 0,
    "discount_percent": (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100),
}

for c, cond in invalid_rules.items():
    bad = df.filter(cond)
    print(f"{c}: {bad.count()} invalid records")
    bad.select("order_id", c).show(5)

quantity: 12 invalid records
+--------+--------+
|order_id|quantity|
+--------+--------+
|  100114|       0|
|  100143|       0|
|  100198|      -2|
|  100365|       0|
|  100375|       0|
+--------+--------+
only showing top 5 rows
unit_price: 8 invalid records
+--------+----------+
|order_id|unit_price|
+--------+----------+
|  100102|         0|
|  100170|         0|
|  100235|      -500|
|  100284|      -500|
|  100487|      -500|
+--------+----------+
only showing top 5 rows
discount_percent: 8 invalid records
+--------+----------------+
|order_id|discount_percent|
+--------+----------------+
|  100034|              -5|
|  100140|              -5|
|  100217|              -5|
|  100310|             120|
|  100363|             120|
+--------+----------------+
only showing top 5 rows


### 2.4 Inconsistent / invalid dates

In [13]:
iso_pattern = r"^\d{4}-\d{2}-\d{2}$"

# Dates that are not in yyyy-MM-dd shape
df.filter(F.col("order_date").isNotNull() & ~F.col("order_date").rlike(iso_pattern)) \
  .select("order_id", "order_date").show(truncate=False)

+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100358  |2025/07/14|
|100448  |not-a-date|
|100969  |09-15-2026|
+--------+----------+



## 3. Basic preprocessing - Silver layer

Steps: remove exact duplicates -> trim / standardize text -> handle missing text -> validate numbers -> standardize dates.
Records are kept wherever possible; invalid values are fixed or set to NULL instead of deleting rows.

### 3.1 Remove exact duplicates

In [14]:
clean_df = df.dropDuplicates()
rows_after_dedup = clean_df.count()
print("Rows removed as exact duplicates:", bronze_rows - rows_after_dedup)

Rows removed as exact duplicates: 10


### 3.2 Trim spaces and standardize capitalization

In [15]:
# Trim, collapse repeated spaces, and turn blank strings into NULL
for c in text_cols:
    clean_df = clean_df.withColumn(c, F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))
    clean_df = clean_df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))

# Standardize capitalization only where it is safe.
# 'product' and 'payment_method' are left alone: initcap would damage values
# such as 'Cooking Oil 5L', 'T-Shirt', 'USB-C Hub' and 'UPI'.
for c in ["customer_name", "category", "city", "state", "order_status"]:
    clean_df = clean_df.withColumn(c, F.initcap(F.col(c)))

# Trimming/standardizing can reveal duplicates that were hidden by spacing/case
before = clean_df.count()
clean_df = clean_df.dropDuplicates()
print("Extra duplicates found after standardizing text:", before - clean_df.count())

Extra duplicates found after standardizing text: 0


### 3.3 Handle missing text values

In [16]:
missing_text_cols = ["customer_name", "product", "category", "payment_method",
                     "city", "state", "order_status"]

clean_df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in missing_text_cols]).show()

clean_df = clean_df.fillna({c: "Unknown" for c in missing_text_cols})

+-------------+-------+--------+--------------+----+-----+------------+
|customer_name|product|category|payment_method|city|state|order_status|
+-------------+-------+--------+--------------+----+-----+------------+
|           10|      0|       0|             8|   8|    0|           0|
+-------------+-------+--------+--------------+----+-----+------------+



### 3.4 Validate quantity, unit price and discount percentage
Invalid values (quantity/price <= 0, discount outside 0-100) and missing values are replaced with the **median of the valid values**, so the record is preserved.

In [17]:
valid_rules = {
    "quantity":         F.col("quantity") > 0,
    "unit_price":       F.col("unit_price") > 0,
    "discount_percent": (F.col("discount_percent") >= 0) & (F.col("discount_percent") <= 100),
}

medians = {}
for c, valid in valid_rules.items():
    to_fix = clean_df.filter(~F.coalesce(valid, F.lit(False))).count()   # invalid or NULL
    clean_df = clean_df.withColumn(c, F.when(valid, F.col(c)))           # invalid -> NULL
    medians[c] = clean_df.approxQuantile(c, [0.5], 0.0)[0]               # median of valid values only
    clean_df = clean_df.fillna({c: int(round(medians[c]))})
    print(f"{c}: {to_fix} values replaced with median {medians[c]}")

quantity: 12 values replaced with median 5.0
unit_price: 8 values replaced with median 1850.0
discount_percent: 8 values replaced with median 20.0


### 3.5 Standardize dates

In [18]:
date_formats = ["yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy", "dd-MM-yyyy", "MM/dd/yyyy", "MM-dd-yyyy"]

# try_to_timestamp returns NULL (instead of failing) when a value does not match / is not a real date.
# Day-first formats are tried before month-first ones.
parsed_date = F.coalesce(*[
    F.to_date(F.try_to_timestamp(F.trim(F.col("order_date")), F.lit(fmt)))
    for fmt in date_formats
])

clean_df = clean_df.withColumn("order_date_parsed", parsed_date)

invalid_dates = clean_df.filter(F.col("order_date_parsed").isNull() & F.col("order_date").isNotNull())
print("Invalid / unparseable dates:", invalid_dates.count())
invalid_dates.select("order_id", "order_date").show(truncate=False)

Invalid / unparseable dates: 4
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100637  |2026-13-05|
|100448  |not-a-date|
|100634  |2025-02-30|
+--------+----------+



In [19]:
# Invalid dates are kept as NULL rather than deleting the record
clean_df = (clean_df.drop("order_date")
                    .withColumnRenamed("order_date_parsed", "order_date")
                    .select(df.columns))        # keep the original column order

clean_df.printSchema()
clean_df.show(10, truncate=False)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- unit_price: integer (nullable = false)
 |-- discount_percent: integer (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+------------

### 3.6 Save to the Silver layer

In [20]:
silver_path = "data_lake/silver/preprocessed_sales"

# Parquet (Spark output)
clean_df.write.mode("overwrite").parquet(silver_path)

# Single CSV file for the GitHub submission (silver/sales_silver.csv)
clean_df.toPandas().to_csv("data_lake/silver/sales_silver.csv", index=False)

show_tree()

data_lake/
    bronze/
        sales.csv
    silver/
        sales_silver.csv
        preprocessed_sales/
            _SUCCESS
            part-00000-2a5d50ab-c41c-4b5a-a70f-8cf8058e00e8-c000.snappy.parquet


In [21]:
# Download the CSV (only works inside Colab)
try:
    from google.colab import files
    files.download("data_lake/silver/sales_silver.csv")
except ImportError:
    pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Verification

In [22]:
silver_df = spark.read.parquet(silver_path)
silver_rows = silver_df.count()

print("Rows before preprocessing (Bronze):", bronze_rows)
print("Rows after preprocessing  (Silver):", silver_rows)
print("Rows removed                      :", bronze_rows - silver_rows)

silver_df.printSchema()
silver_df.show(10, truncate=False)

Rows before preprocessing (Bronze): 1000
Rows after preprocessing  (Silver): 990
Rows removed                      : 10
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |sta

In [23]:
# Remaining NULLs per column (only invalid dates should remain NULL)
silver_df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in silver_df.columns]).show()

# Re-check the validation rules on the Silver data
silver_df.select(
    F.min("quantity").alias("min_quantity"),
    F.min("unit_price").alias("min_unit_price"),
    F.min("discount_percent").alias("min_discount"),
    F.max("discount_percent").alias("max_discount"),
    F.min("order_date").alias("earliest_date"),
    F.max("order_date").alias("latest_date"),
).show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|         4|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

+------------+--------------+------------+------------+-------------+-----------+
|min_quantity|min_unit_price|min_discount|max_discount|earliest_date|latest_date|
+------------+--------------+------------+------------+-------------+-----------+
|           1|           190|           0|      

In [24]:
# Confirm the Bronze file is byte-for-byte identical to the original upload
import hashlib

def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

print("Bronze unchanged:", md5("sales.csv") == md5("data_lake/bronze/sales.csv"))

Bronze unchanged: True


## Preprocessing operations performed

1. Kept the raw `sales.csv` unchanged in Bronze (verified with a checksum).
2. Inspected duplicates, nulls, text formatting, numeric ranges and date formats using Spark.
3. Removed exact duplicate rows (and re-checked for duplicates after text standardization).
4. Trimmed leading/trailing spaces, collapsed repeated spaces, and converted blank strings to NULL.
5. Standardized capitalization of names, category, city, state and order status (product and payment method were left as-is to avoid damaging values like `5L`, `T-Shirt`, `UPI`).
6. Filled missing text values with `"Unknown"`.
7. Treated quantity <= 0, unit price <= 0 and discount outside 0-100 as invalid, and replaced them (and missing numbers) with the median of the valid values.
8. Converted dates in several formats to one format (`yyyy-MM-dd`); impossible/unparseable dates were kept as NULL instead of deleting the record.
9. Saved the Silver data as Parquet and as a single CSV, then read it back with Spark to verify row counts, schema and sample rows.

**Note:** `order_id` is not unique in the raw data (different customers/products share some IDs). These are not exact duplicates, so the records were preserved.